<a href="https://colab.research.google.com/github/Victand/api_purchase_predict/blob/staging/InstantMesh_jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-

import os
import shutil

# =========================================================
# 1. PRÉPARATION ET INSTALLATION DES DÉPENDANCES
# =========================================================

%cd /content

# Nettoyage en cas de relance
if os.path.exists('/content/InstantMesh'):
    shutil.rmtree('/content/InstantMesh')

# Clonage de VOTRE fork sur la branche diffusion_swap
!GIT_LFS_SKIP_SMUDGE=1 git clone -b diffusion_swap https://github.com/SihamDaanouni/InstantMesh
%cd /content/InstantMesh

# Vérification qu'on est sur la bonne branche avec le bon run.py
!git branch
!echo "--- Vérification run.py (doit contenir diffusion_model) ---"
!grep -n "diffusion_model" run.py | head -5

# Clonage de SyncDreamer
%cd /content
if os.path.exists('/content/SyncDreamer'):
    shutil.rmtree('/content/SyncDreamer')
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/liuyuan-pal/SyncDreamer

# 1. Fix numpy AVANT tout
!pip install "numpy<2.0.0"

# 2. GPU et utilitaires 3D
!pip install onnxruntime-gpu ninja
!pip install PyMCubes trimesh rembg xatlas plyfile imageio[ffmpeg] jax==0.4.19 jaxlib==0.4.19

# 3. Machine Learning
!pip install pytorch-lightning==2.1.2 gradio==3.50.2 einops omegaconf torchmetrics webdataset accelerate tensorboard

# 4. HuggingFace
!pip install "huggingface-hub<1.0" "transformers>=4.41.0" "diffusers>=0.27.0"

# 5. Variables CUDA
os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['PATH'] += ':/usr/local/cuda/bin'
os.environ['LD_LIBRARY_PATH'] += ':/usr/local/cuda/lib64'

# 6. nvdiffrast
%cd /content
if os.path.exists('/content/nvdiffrast'):
    shutil.rmtree('/content/nvdiffrast')
!git clone https://github.com/NVlabs/nvdiffrast.git
%cd /content/nvdiffrast
!pip install --no-build-isolation .

%cd /content/InstantMesh
print("\n=== INSTALLATION TERMINÉE ===")
print("Branche active :", end=" ")
!git branch --show-current

In [ ]:
!pip install "cupy-cuda12x<14.0.0"

In [ ]:
!pip uninstall -y flax

In [ ]:
!pip install --upgrade transformers==4.41.0 diffusers==0.27.0 jax==0.4.23 jaxlib==0.4.23

In [ ]:
!pip install huggingface_hub==0.25.2

In [ ]:
!pip install peft==0.10.0

In [ ]:
# Download syncdreamer checkpoint
!pip install -q gdown

import gdown
import os

CKPT_URL = "https://drive.google.com/uc?id=1ypyD5WXxAnsWjnHgAfOAGolV0Zd9kpam"
output_dir = "/content/SyncDreamer/ckpt"

# create ckpt dir
os.makedirs(output_dir, exist_ok=True)

# download
gdown.download(CKPT_URL, os.path.join(output_dir, "syncdreamer-pretrain.ckpt"), quiet=False)


In [ ]:
# =========================================================
# 2. INFÉRENCE — Référence originale (fine-tuned)
# =========================================================
%cd /content/InstantMesh

!python run.py configs/instant-mesh-base.yaml examples/hatsune_miku.png \
    --save_video \
    --diffusion_model zero123plus_finetuned

print("Référence terminée")
print("outputs/instant-mesh-base_zero123plus_finetuned/")

In [ ]:
# =========================================================
# 3. INFÉRENCE — Option A : sans fine-tuning
# =========================================================
%cd /content/InstantMesh

!python run.py configs/instant-mesh-base.yaml examples/hatsune_miku.png \
    --save_video \
    --diffusion_model zero123plus_base

print("Option A : modèle sans fine-tuning terminée")
print("outputs/instant-mesh-base_zero123plus_base/")

In [ ]:
# =========================================================
# 3. INFÉRENCE — Option B : sans fine-tuning
# =========================================================
%cd /content/InstantMesh

!python run.py configs/instant-mesh-base.yaml examples/hatsune_miku.png \
    --save_video \
    --diffusion_model syncdreamer

print("Option A : modèle sans fine-tuning terminée")
print("outputs/instant-mesh-base_zero123plus_base/")

In [ ]:
# =========================================================
# 4. VISUALISATION
# =========================================================
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Video
import os

# --- Vues multi-vues ---
paths = {
    "Original (fine-tuned)": "outputs/instant-mesh-base_zero123plus_finetuned/images/hatsune_miku.png",
    "Option A (sans fine-tuning)":       "outputs/instant-mesh-base_zero123plus_base/images/hatsune_miku.png",
}

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for ax, (label, path) in zip(axes, paths.items()):
    if os.path.exists(path):
        ax.imshow(np.array(Image.open(path)))
        ax.set_title(label, fontsize=13, fontweight='bold')
    else:
        # Remplacement de l'émoji par du texte pour éviter l'erreur Glyph
        ax.set_title(f"!!! FICHIER MANQUANT !!!\n{path}", color='red', fontsize=10)
        print(f"Attention : Image absente -> {path}")
    ax.axis('off')

plt.suptitle("Comparaison des 6 vues générées", fontsize=15, fontweight='bold')
plt.tight_layout()
os.makedirs('outputs/comparaisons', exist_ok=True)
plt.savefig('outputs/comparaisons/vues_comparaison.png', dpi=150)
plt.show()

# --- Vidéos 3D ---
video_paths = {
    "Original": "outputs/instant-mesh-base_zero123plus_finetuned/videos/hatsune_miku.mp4",
    "Option A":  "outputs/instant-mesh-base_zero123plus_base/videos/hatsune_miku.mp4",
}

for label, path in video_paths.items():
    if os.path.exists(path):
        print(f"\n[OK] Affichage : {label}")
        display(Video(path, embed=True, width=512))
    else:
        # Utilisation de [X] au lieu de l'émoji pour la console si besoin
        print(f"[X] Vidéo manquante : {path}")

In [ ]:
!pip install lpips scikit-image "numpy==1.26.4" "scipy<1.13.0"

In [ ]:
import torch
import numpy as np
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import lpips
import pandas as pd
import os
%cd /content/InstantMesh

# Initialisation du modèle LPIPS
loss_fn = lpips.LPIPS(net='alex')

def load_6views(path):
    img = np.array(Image.open(path).convert('RGB'), dtype=np.float32) / 255.0
    H, W = img.shape[0] // 3, img.shape[1] // 2
    views = []
    for row in range(3):
        for col in range(2):
            views.append(img[row*H:(row+1)*H, col*W:(col+1)*W])
    return np.stack(views)

def compute_metrics(path_ref, path_other):
    if not os.path.exists(path_ref) or not os.path.exists(path_other):
        return {'PSNR ↑': 'Erreur', 'SSIM ↑': 'Erreur', 'LPIPS ↓': f'Fichier manquant'}

    v_ref, v_other = load_6views(path_ref), load_6views(path_other)
    psnr_l, ssim_l, lpips_l = [], [], []
    for i in range(6):
        psnr_l.append(psnr(v_ref[i], v_other[i], data_range=1.0))
        ssim_l.append(ssim(v_ref[i], v_other[i], channel_axis=2, data_range=1.0))

        t1 = torch.from_numpy(v_ref[i]).permute(2,0,1).unsqueeze(0) * 2 - 1
        t2 = torch.from_numpy(v_other[i]).permute(2,0,1).unsqueeze(0) * 2 - 1
        lpips_l.append(loss_fn(t1, t2).item())

    return {'PSNR ↑': round(np.mean(psnr_l),3),
            'SSIM ↑': round(np.mean(ssim_l),4),
            'LPIPS ↓': round(np.mean(lpips_l),4)}

NOM_IMAGE = "hatsune_miku.png"

# Utilisation des CHEMINS ABSOLUS pour éviter que Colab se perde
path_ref  = f"/content/InstantMesh/outputs/instant-mesh-base_zero123plus_finetuned/images/{NOM_IMAGE}"
path_base = f"/content/InstantMesh/outputs/instant-mesh-base_zero123plus_base/images/{NOM_IMAGE}"

print("Calcul des métriques en cours...")
df = pd.DataFrame([
    {"Modèle": "Original fine-tuned (référence)", **compute_metrics(path_ref, path_ref)},
    {"Modèle": "Option A — base sans fine-tuning", **compute_metrics(path_ref, path_base)},
]).set_index("Modèle")

print("\n", df.to_string())

os.makedirs('/content/InstantMesh/outputs/comparaisons', exist_ok=True)
df.to_csv('/content/InstantMesh/outputs/comparaisons/metriques.csv')
print("\n Sauvegardé dans outputs/comparaisons/metriques.csv")